In [1]:
import torch
from torch import nn
import torchvision
from torch.utils import data
from torchvision import datasets, transforms
from d2l import torch as d2l
from torch.nn import functional as F
import random
import numpy as np
seed=42
batch_size=64
epoch=20
lr=0.05
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 让CUDA尽可能使用确定性算法
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False


set_seed(seed)


In [ ]:
def data_maker():
    mean = (
        0.4914,
        0.4822,
        0.4465
    )

    std = (
        0.2470,
        0.2435,
        0.2616
    )

    train_transform = transforms.Compose([
        # 填充成40×40，再随机裁剪回32×32
        transforms.RandomCrop(
            size=32,
            padding=4
        ),

        # 50%概率水平翻转
        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        # 轻微改变颜色
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.05
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=mean,
            std=std
        )
    ])

    test_transform = transforms.Compose([
        transforms.ToTensor(),

        # 测试集必须使用同样的标准化参数
        transforms.Normalize(
            mean=mean,
            std=std
        )
    ])

    train_dataset = torchvision.datasets.CIFAR10(
        root="../cifar10-data",
        train=True,
        transform=train_transform,
        download=False
    )

    test_dataset = torchvision.datasets.CIFAR10(
        root="../cifar10-data",
        train=False,
        transform=test_transform,
        download=False
    )

    return train_dataset, test_dataset

In [ ]:
def data_loader(dataset,batch_size,flag):
    return data.DataLoader(dataset,batch_size=batch_size,shuffle=flag, num_workers=0,
        pin_memory=True,
        )


In [4]:
train_dataset,test_dataset=data_maker()
print(train_dataset.targets)


[6, 9, 9, 4, 1, 1, 2, 7, 8, 3, 4, 7, 7, 2, 9, 9, 9, 3, 2, 6, 4, 3, 6, 6, 2, 6, 3, 5, 4, 0, 0, 9, 1, 3, 4, 0, 3, 7, 3, 3, 5, 2, 2, 7, 1, 1, 1, 2, 2, 0, 9, 5, 7, 9, 2, 2, 5, 2, 4, 3, 1, 1, 8, 2, 1, 1, 4, 9, 7, 8, 5, 9, 6, 7, 3, 1, 9, 0, 3, 1, 3, 5, 4, 5, 7, 7, 4, 7, 9, 4, 2, 3, 8, 0, 1, 6, 1, 1, 4, 1, 8, 3, 9, 6, 6, 1, 8, 5, 2, 9, 9, 8, 1, 7, 7, 0, 0, 6, 9, 1, 2, 2, 9, 2, 6, 6, 1, 9, 5, 0, 4, 7, 6, 7, 1, 8, 1, 1, 2, 8, 1, 3, 3, 6, 2, 4, 9, 9, 5, 4, 3, 6, 7, 4, 6, 8, 5, 5, 4, 3, 1, 8, 4, 7, 6, 0, 9, 5, 1, 3, 8, 2, 7, 5, 3, 4, 1, 5, 7, 0, 4, 7, 5, 5, 1, 0, 9, 6, 9, 0, 8, 7, 8, 8, 2, 5, 2, 3, 5, 0, 6, 1, 9, 3, 6, 9, 1, 3, 9, 6, 6, 7, 1, 0, 9, 5, 8, 5, 2, 9, 0, 8, 8, 0, 6, 9, 1, 1, 6, 3, 7, 6, 6, 0, 6, 6, 1, 7, 1, 5, 8, 3, 6, 6, 8, 6, 8, 4, 6, 6, 1, 3, 8, 3, 4, 1, 7, 1, 3, 8, 5, 1, 1, 4, 0, 9, 3, 7, 4, 9, 9, 2, 4, 9, 9, 1, 0, 5, 9, 0, 8, 2, 1, 2, 0, 5, 6, 3, 2, 7, 8, 8, 6, 0, 7, 9, 4, 5, 6, 4, 2, 1, 1, 2, 1, 5, 9, 9, 0, 8, 4, 1, 1, 6, 3, 3, 9, 0, 7, 9, 7, 7, 9, 1, 5, 1, 6, 6, 8, 7, 1, 3, 0, 

In [5]:
def try_gpu(i=0):
    if torch.cuda.device_count() >= i+1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')
device=try_gpu()

In [6]:
class residual_normal(nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.net=nn.Sequential()
        self.net.append(nn.Conv2d(in_channel,out_channel,kernel_size=3,stride=1,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
        self.net.append(nn.ReLU())
        self.net.append(nn.Conv2d(out_channel,out_channel,kernel_size=3,stride=1,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
    def forward(self,x):
        y=self.net(x)
        y=y+x
        return F.relu(y)
    
class residual(nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.net=nn.Sequential()
        self.net.append(nn.Conv2d(in_channel,out_channel,kernel_size=3,stride=2,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
        self.net.append(nn.ReLU())
        self.net.append(nn.Conv2d(out_channel,out_channel,kernel_size=3,stride=1,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
        self.net1 = nn.Sequential(
    nn.Conv2d(
        in_channel,
        out_channel,
        kernel_size=1,
        stride=2,
        bias=False
    ),
    nn.BatchNorm2d(out_channel)
)
    def forward(self,x):
        y=self.net(x)+self.net1(x)
        return F.relu(y)

In [7]:
def net_maker():
    b1 = nn.Sequential(
        nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        ),
        nn.BatchNorm2d(64),
        nn.ReLU()
    )

    b2 = nn.Sequential(
        residual_normal(64, 64),
        residual_normal(64, 64)
    )

    b3 = nn.Sequential(
        residual(64, 128),
        residual_normal(128, 128)
    )

    b4 = nn.Sequential(
        residual(128, 256),
        residual_normal(256, 256)
    )

    b5 = nn.Sequential(
        residual(256, 512),
        residual_normal(512, 512)
    )

    net = nn.Sequential(
        b1,
        b2,
        b3,
        b4,
        b5,
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Linear(512, 10)
    )

    return net

In [8]:
net=net_maker().to(device)
train_iter=data_loader(train_dataset,batch_size,True)
test_iter=data_loader(test_dataset,batch_size,False)
trainer=torch.optim.SGD(net.parameters(),lr=lr,momentum=0.9,
    weight_decay=5e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(trainer,T_max=epoch,eta_min=1e-4)
loss=nn.CrossEntropyLoss()


In [9]:
import time

print("CUDA可用：", torch.cuda.is_available())
print("指定设备：", device)
print("模型设备：", next(net.parameters()).device)

safe_test_iter = data.DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

start = time.time()

x, y = next(iter(safe_test_iter))

print("读取一个批次耗时：", time.time() - start)
print("x形状：", x.shape)

CUDA可用： True
指定设备： cuda:0
模型设备： cuda:0
读取一个批次耗时： 0.08567166328430176
x形状： torch.Size([512, 3, 32, 32])


In [10]:
torch.cuda.synchronize()

start = time.time()

net.eval()

with torch.no_grad():
    logit = net(x.to(
        device,
        non_blocking=True
    ))

torch.cuda.synchronize()

print(
    "512张图片前向传播耗时：",
    time.time() - start
)

print("输出形状：", logit.shape)

512张图片前向传播耗时： 4.585759878158569
输出形状： torch.Size([512, 10])


In [11]:
for i in range(epoch):
    net.train()
    train_loss=0.0
    train_count=0
    train_acc=0
    for x,y in train_iter:
        x=x.to(device,non_blocking=True)
        y=y.to(device,non_blocking=True)
        trainer.zero_grad()
        logit=net(x)
        l=loss(logit,y)
        l.backward()
        trainer.step()
        train_loss+=l.item()*len(y)
        train_count+=len(y)
        train_acc+=(logit.argmax(dim=1)==y).sum().item()
    scheduler.step()
    print(f"第{i+1}个epoch的train损失:{train_loss/train_count:.5f},acc:{train_acc/train_count:.5f}")
    net.eval()
    with torch.no_grad():
        test_loss=0.0
        test_count=0
        test_acc=0
        for x,y in test_iter:
            x=x.to(device,non_blocking=True)
            y=y.to(device,non_blocking=True)
            logit=net(x)
            l=loss(logit,y)
            test_loss+=l.item()*len(y)
            test_count+=len(y)
            test_acc+=(logit.argmax(dim=1)==y).sum().item()
        print(f"第{i+1}个epoch的test损失:{test_loss/test_count:.5f},acc:{test_acc/test_count:.5f}")



第1个epoch的train损失:1.75194,acc:0.35688
第1个epoch的test损失:1.42982,acc:0.47450
第2个epoch的train损失:1.22625,acc:0.55760
第2个epoch的test损失:1.06483,acc:0.61680
第3个epoch的train损失:0.93229,acc:0.67074
第3个epoch的test损失:0.85002,acc:0.70710
第4个epoch的train损失:0.75559,acc:0.73646
第4个epoch的test损失:0.74884,acc:0.75000
第5个epoch的train损失:0.65112,acc:0.77408
第5个epoch的test损失:0.68997,acc:0.76410
第6个epoch的train损失:0.58063,acc:0.79950
第6个epoch的test损失:0.59149,acc:0.79710
第7个epoch的train损失:0.52830,acc:0.81758
第7个epoch的test损失:0.53566,acc:0.81410
第8个epoch的train损失:0.47923,acc:0.83606
第8个epoch的test损失:0.50466,acc:0.83040
第9个epoch的train损失:0.44766,acc:0.84698
第9个epoch的test损失:0.51433,acc:0.82270
第10个epoch的train损失:0.41067,acc:0.85904
第10个epoch的test损失:0.41846,acc:0.85620
第11个epoch的train损失:0.37497,acc:0.87044
第11个epoch的test损失:0.46482,acc:0.84710
第12个epoch的train损失:0.33994,acc:0.88264
第12个epoch的test损失:0.41852,acc:0.85860
第13个epoch的train损失:0.30155,acc:0.89650
第13个epoch的test损失:0.34288,acc:0.88270
第14个epoch的train损失:0.26679,acc:0.90870
第14个e